In [ ]:
import pandas as pd
from feature_extraction_utils import load_images, image_label_pred, download_model

classifier = download_model(model = r"openai/clip-vit-base-patch32")

label_list = ["with a license plate", "without a license plate"]
color_list = ["white", "black", "red", "orange", "yellow", "green", "blue", "purple"]
vehicle_list = ["car", "bus", "motorcycle", "license plate"]

color_labels = []

for color in color_list:
    text_string = f"a {color} vehicle"
    color_labels.append(text_string)

vehicle_labels = []
for vehicle in vehicle_list:
    text_string = f"a {vehicle}"
    vehicle_labels.append(text_string)

image_folder = r"../data/formatted/license_plate_detection/train/images"
image_set = load_images(directory= image_folder, num_img= 500, use_rand= True, img_obj= True)

# prob_list = []
# for idx, image in enumerate(image_set):
#     df = image_label_pred(image[0], vehicle_list, classifier)
    
#     df["fn"] = image[1]
#     prob_list.append(df)
#     pct_complete = (idx+1)/len(image_set)
#     print(f"%{pct_complete*100:.1f}")

pil_images = [img[0] for img in image_set]
file_names = [img[1] for img in image_set]
batch_predictions = classifier(pil_images, candidate_labels=vehicle_list, batch_size=8)
    
prob_list = []

# 3. Iterate through the results to format your DataFrames
for idx, (predictions, file_name) in enumerate(zip(batch_predictions, file_names)):
    df = pd.DataFrame(predictions).T
    df.columns = df.iloc[-1]
    df = df[:-1]
    df["fn"] = file_name
    prob_list.append(df)
    
    pct_complete = (idx + 1) / len(image_set)
    print(f"%{pct_complete*100:.2f} formatted")
    
    



In [ ]:
import pandas as pd
result = pd.concat(prob_list)

result.to_csv("C:/Users/Installer/Downloads/vehicle_labels.csv")

df = pd.read_csv("C:/Users/Installer/Downloads/vehicle_labels.csv")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
df_license_plate_only = df[df["license plate"] > .9].reset_index()

cols = 3
n = df_license_plate_only.shape[0]
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize = (12, rows * 3) )

axes_flat = axes.flatten()

for idx, row in df_license_plate_only.iterrows():
    axes_flat[idx].imshow(Image.open(row["fn"]))
    axes_flat[idx].set_title(f'Probability {row["license plate"]:.2f}')